In [1]:
#!/usr/bin/env python3
"""
shuukh_parser.py — turns exported shuukh.mn case files (HTML wearing a .doc
extension) into one structured CSV row per case, across criminal, civil,
and administrative cases.

USAGE
    python3 shuukh_parser.py <input_folder> <output_csv>

WORKFLOW
    1. Drop however many exported case files you've pulled into a folder
       (any extension is fine — .doc, .html, whatever the export gave you).
    2. Run this script against that folder.
    3. Open the CSV. Any row with something in 'parse_warnings' is a case
       the script couldn't confidently read — go look at that file by hand,
       see what's different, and we extend the matching pattern for it.
       Don't trust a field that came back blank — it's blank on purpose,
       not silently guessed.
    4. Add more files, rerun. The CSV is meant to be regenerated from
       scratch each time, not hand-edited.

SCOPE NOTES (as of the last real-data pass)
    - case_level filters out appellate/supervisory rulings (they review a
      PRIOR decision, not a fresh case — wrong shape for this dataset).
      Only 'first_instance' rows get outcome extraction attempted.
    - criminal: solid, built/validated against 5 real cases spanning
      convictions (fine, imprisonment, suspended sentence) and an
      acquittal.
    - civil: PRELIMINARY. classify_civil_outcome and extract_civil_award
      are built from exactly ONE real granted-claim example. The
      'denied'/'partial' verb forms are best guesses, not yet confirmed
      against real cases. Expect civil_outcome_unknown warnings until
      more diverse civil examples come in.
    - administrative: NOT YET MODELED. Every real example seen so far is
      a procedural dismissal (consent/settlement), not a merits ruling,
      so there's no outcome pattern to build from yet. Rows get flagged
      'administrative_outcome_not_yet_modeled' rather than a guess.

WHY FIELDS ARE SCOPED THE WAY THEY ARE
    Court decisions repeat key numbers (fines, court fees, claim amounts)
    several times across the narrative, the reasoning, and the verdict —
    and they aren't all the same number. A naive "find any MNT-looking
    number" regex will find several different ones and you won't know
    which is correct (confirmed in both a criminal case with 13 MNT
    mentions across 3 different real figures, and a civil case where the
    verdict's only MNT figure was a court-fee reimbursement, not the
    claim outcome). So fine_amount, civil_award_amount, and the convicted
    articles are read ONLY from inside the 'ТОГТООХ нь:' (operative
    verdict) section, and within that, specifically from point '1.' —
    which by legal convention is the actual substantive ruling, not
    procedural boilerplate (court fees, appeal rights, etc.) that comes
    after it.
"""

import re
import csv
import sys
from pathlib import Path
from bs4 import BeautifulSoup


def load_text(path: Path) -> str:
    raw = path.read_bytes().decode("utf-8", errors="replace")
    soup = BeautifulSoup(raw, "html.parser")
    text = soup.get_text(separator="\n")
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text


def extract_case_id(text, warnings):
    """
    Two different numbering schemes exist: criminal cases are
    YEAR/CODE/NUM (e.g. 2026/ШЦТ/296), civil/admin cases are
    NUM/CODE+YEAR/NUM (e.g. 192/ШШ2026/06554) — the code and year are
    fused together with no separating slash in the second scheme. Rather
    than hardcode which part is letters vs digits, just require three
    slash-separated alphanumeric segments.
    """
    m = re.search(r"Дугаар\s+([\dА-ЯӨҮ]+(?:/[\dА-ЯӨҮ]+){2})", text)
    if m:
        return m.group(1)
    warnings.append("case_id")
    return None


def extract_decision_date(text, warnings):
    m = re.search(r"(\d{4})\s*оны\s*(\d{1,2})\s*сарын\s*(\d{1,2})\s*өдөр", text)
    if m:
        y, mo, d = m.groups()
        return f"{y}-{int(mo):02d}-{int(d):02d}"
    warnings.append("decision_date")
    return None


def get_case_category(text, warnings):
    """
    Criminal vs civil vs administrative.

    The court-name title isn't reliable on its own — some circuits are
    titled e.g. 'Эрүү, Иргэний хэргийн ... шүүх' because that one court
    handles BOTH criminal and civil simplified-procedure cases, so a
    title-text check can mis-fire. What's reliable is which legal code
    the decision actually cites throughout the body: criminal cases cite
    'Эрүүгийн хуулийн' (Criminal Code) repeatedly, civil cases cite
    'Иргэний хуулийн' (Civil Code) instead, and essentially never both.
    """
    criminal_hits = len(re.findall(r"Эрүүгийн хуулийн", text))
    civil_hits = len(re.findall(r"Иргэний хуулийн", text))
    admin_hits = len(re.findall(r"Захиргааны хэрэг", text[:1500])) + len(
        re.findall(r"Захиргааны хэргийн", text[:1500])
    )

    if criminal_hits and criminal_hits >= civil_hits:
        return "criminal"
    if civil_hits:
        return "civil"
    if admin_hits:
        return "administrative"
    warnings.append("case_category_unknown")
    return None


def get_case_level(text):
    """
    First instance vs appellate vs supervisory (cassation).

    An appellate/supervisory ruling reviews a PRIOR court's decision —
    it upholds, reverses, or remands. That's a fundamentally different
    prediction target than 'decide this case fresh', so these need to be
    filtered out of the training table, not force-fit into the same
    outcome schema. Confirmed real example: a Supreme Court 'Шүүх
    хуралдааны тогтоол' reviewing a civil case on cassation — wrong
    shape entirely for this dataset.
    """
    head = text[:500]
    if "Улсын дээд шүүх" in head or "хяналтын журмаар" in text[:3000]:
        return "supervisory"
    if "давж заалдах шатны шүүх" in head:
        return "appellate"
    return "first_instance"


def get_point1_text(verdict_text):
    """
    Isolate point '1.' of the operative verdict.

    The boundary for 'where point 1 ends' can't just be the next literal
    '2.' in the text — article numbers like '2.8' contain '2.' too, and
    a naive search stops there instead of at the real list marker. A real
    list point starts at a line break, optional whitespace, the number,
    a dot, then a SPACE and a capital letter (the start of a new
    sentence) — '2.8' is immediately followed by another digit, not a
    space+capital, so this pattern doesn't false-trigger on it.

    \\s* around the digit/period (rather than assuming '1.' is two
    adjacent characters) handles exports that insert a stray line break
    between the number and the dot.
    """
    if verdict_text is None:
        return None
    start = re.search(r"^\s*1\s*\.\s*", verdict_text)
    if not start:
        return None
    rest = verdict_text[start.end():]
    end = re.search(r"\n\s*\d+\s*\.\s*[А-ЯӨҮ]", rest)
    return rest[:end.start()] if end else rest


def classify_outcome(point1_text, warnings):
    """Read the actual outcome verb from point 1 — never trust the
    document title, it's just a generic template name at some courts."""
    if point1_text is None:
        warnings.append("outcome_no_point1")
        return None
    if re.search(r"гэм буруутайд тооцсугай|гэм буруутай болохыг тогтоо", point1_text):
        return 1
    if re.search(r"цагаатгасугай|цагаатгах тогтоол", point1_text):
        return 0
    if re.search(r"хэрэгсэхгүй болго", point1_text):
        return 0
    warnings.append("outcome_unknown")
    return None


def extract_articles(point1_text, warnings):
    """Special-part (тусгай анги = actual crime) articles only, scoped to
    point 1. Classifies each article mention by whichever qualifier —
    'тусгай ангийн' (special) or 'ерөнхий ангийн' (general/procedural) —
    appears closest before it, since general-part procedure articles
    (sentencing rules, etc.) get cited in the same sentence and must not
    end up in primary/secondary_article."""
    if point1_text is None:
        warnings.append("articles_no_point1")
        return None, None, None

    quals = [(m.start(), "special") for m in re.finditer(r"тусгай ангийн", point1_text)]
    quals += [(m.start(), "general") for m in re.finditer(r"ерөнхий ангийн", point1_text)]
    quals.sort()

    articles = []
    for am in re.finditer(r"(\d+\.\d+)\s*д(?:угаар|үгээр) зүйлийн\s*(\d+)\s*дэх хэсэг", point1_text):
        nearest = None
        for qpos, qtype in quals:
            if qpos < am.start():
                nearest = qtype
            else:
                break
        if nearest == "special":
            code = f"{am.group(1)}.{am.group(2)}"
            if code not in articles:
                articles.append(code)

    if not articles:
        warnings.append("articles")
        return None, None, None
    primary = articles[0]
    secondary = articles[1] if len(articles) > 1 else None
    extra = "|".join(articles[2:]) if len(articles) > 2 else None
    return primary, secondary, extra


def classify_civil_outcome(point1_text, warnings):
    """
    Granted / denied / partial, read from point 1's verb — same pattern
    as criminal classify_outcome.

    LIMITATION: built from exactly one real example so far (a fully
    granted property-title claim). The verb forms for 'denied' and
    'partially granted' haven't been seen in real data yet and are
    guesses — treat 'civil_outcome_unknown' warnings as expected until
    a few denied-claim examples come in to confirm or fix these.
    """
    if point1_text is None:
        warnings.append("civil_outcome_no_point1")
        return None
    if re.search(r"тогтоож|хүлээн зөвшөөрсүгэй|даалгасугай|олгосугай|хангасугай", point1_text):
        return "granted"
    if re.search(r"татгалзсугай|хэрэгсэхгүй болгосугай", point1_text):
        return "denied"
    warnings.append("civil_outcome_unknown")
    return None


def extract_civil_award(point1_text, warnings):
    """
    Monetary amount, but ONLY if point 1 (the actual relief) states one
    directly — never from point 2 onward.

    This is the civil equivalent of the fine_amount leakage trap: a real
    example showed point 2 stating a court-fee reimbursement (улсын
    тэмдэгтийн хураамж) of 630,150 MNT that has NOTHING to do with the
    claim outcome — point 1 itself (property ownership) had no monetary
    amount at all. A naive 'grab any MNT figure from the verdict' would
    have wrongly extracted the fee as if it were the case's outcome.

    Returning None here is not a parse failure — most relief (property
    title, custody, contract performance) isn't monetary at all. Only
    debt/damages/compensation claims will have a real number here.
    """
    if point1_text is None:
        return None
    if "тэмдэгтийн хураамж" in point1_text:
        warnings.append("civil_award_fee_language_in_point1")
        return None
    m = re.search(r"([\d,]{4,})\s*төгрөг", point1_text)
    if m:
        return int(m.group(1).replace(",", ""))
    return None


def extract_bio(text, warnings):
    m = re.search(r"(\d+)\s*настай,\s*(эрэгтэй|эмэгтэй),\s*([^,]+боловсролтой)", text)
    if m:
        return int(m.group(1)), m.group(2), m.group(3).strip()
    warnings.append("bio")
    return None, None, None


def extract_prior_record(text, warnings):
    if re.search(r"эрүүгийн хариуцлага хүлээж байга[а]?г?ү?й?", text):
        return 0
    if re.search(r"эрүүгийн хариуцлага хүлээж байсан", text):
        return 1
    warnings.append("prior_record")
    return None


def extract_simplified_procedure(text):
    return 1 if "хялбаршуулсан журмаар" in text else 0


def get_verdict_section(text):
    """Everything from 'ТОГТООХ нь:' to the end — the operative section.
    Some exports insert a stray line break between 'нь' and ':' (an HTML
    span-wrapping artifact) — \\s* handles that instead of requiring them
    adjacent."""
    m = re.search(r"ТОГТООХ\s*нь\s*:(.*)", text, re.S)
    return m.group(1) if m else None


def get_case_narrative(text, warnings):
    """
    The actual case content — what happened — as a single text blob for
    later NLP/embedding features.

    Window: from 'Хэргийн товч агуулга:' (prosecutor's initial narrative,
    criminal cases) — falling back to 'ТОДОРХОЙЛОХ нь:' (the court's
    factual findings, used across criminal AND civil decisions) when the
    first marker isn't present — through to whichever comes first of
    'Эрүүгийн хариуцлагын талаар:' (sentencing rationale) or 'ТОГТООХ
    нь:' (the verdict).

    Both of those later sections are excluded on purpose: they're where
    the fine amount and final outcome get restated. Including them as an
    input feature would leak the target straight into the text field.
    """
    start = re.search(r"Хэргийн товч агуулга:", text)
    if not start:
        start = re.search(r"ТОДОРХОЙЛОХ\s*нь\s*:", text)
    if not start:
        warnings.append("narrative_no_start")
        return None

    end_candidates = []
    for pattern in [r"Эрүүгийн хариуцлагын талаар:", r"ТОГТООХ нь:"]:
        m = re.search(pattern, text)
        if m and m.start() > start.end():
            end_candidates.append(m.start())

    if not end_candidates:
        warnings.append("narrative_no_end")
        return None

    end = min(end_candidates)
    chunk = text[start.end():end]
    # collapse to a single paragraph — newlines here are just hard line
    # wraps from the original export, not real sentence/paragraph breaks
    chunk = re.sub(r"\s+", " ", chunk).strip()
    return chunk if chunk else None


def classify_punishment_type(verdict_text, is_penalized, warnings):
    """
    Mongolian criminal punishments aren't only fines — imprisonment,
    suspended/probated sentences with activity restrictions, and other
    sanctions all show up too (confirmed directly in a real case: a
    workplace-death conviction punished with a suspended sentence plus an
    occupational ban, zero MNT mentioned anywhere in the verdict).

    This matters because Task 2 (fine_amount regression) only makes sense
    for the subset of convictions that are actually fines. A non-fine
    conviction is NOT 'fined 0' — it's a different outcome entirely, and
    encoding it as fine_amount=0 would teach a regression model that
    convictions often carry zero monetary penalty, which isn't true; they
    just weren't fines.
    """
    if is_penalized == 0:
        return "none"
    if verdict_text is None:
        warnings.append("punishment_type_no_verdict")
        return None
    if re.search(r"төгрөгөөр торгох ял|тэнцэх хэмжээний буюу", verdict_text):
        return "fine"
    if re.search(r"хорих ял оногдуулахгүйгээр", verdict_text):
        return "suspended"
    if re.search(r"хорих ял", verdict_text):
        return "imprisonment"
    warnings.append("punishment_type_unclear")
    return "other"


def extract_fine_amount(verdict_text, punishment_type, warnings):
    if punishment_type == "none":
        return 0
    if punishment_type != "fine":
        return None  # not applicable — a different punishment type, not a $0 fine
    if verdict_text is None:
        warnings.append("fine_amount_no_verdict_section")
        return None
    # Multi-charge cases: the combined total is explicitly labeled "нийт
    # эдлэх ялыг" (total sentence to be served).
    m = re.search(
        r"нийт эдлэх ялыг\s*[\d\s]+нэгжтэй тэнцэх хэмжээний буюу\s*([\d,]+)\s*төгрөг",
        verdict_text,
    )
    if m:
        return int(m.group(1).replace(",", ""))
    # Single-charge cases won't have a "нийт" total to sum — fall back to
    # the plain fine phrase, still scoped to the verdict section only.
    m = re.search(r"тэнцэх хэмжээний буюу\s*([\d,]+)\s*төгрөгөөр торгох ял", verdict_text)
    if m:
        return int(m.group(1).replace(",", ""))
    warnings.append("fine_amount")
    return None


def parse_file(path: Path) -> dict:
    text = load_text(path)
    warnings = []

    case_id = extract_case_id(text, warnings)
    decision_date = extract_decision_date(text, warnings)
    category = get_case_category(text, warnings)
    level = get_case_level(text)

    row = {
        "source_file": path.name,
        "case_id": case_id,
        "decision_date": decision_date,
        "case_category": category,
        "case_level": level,
        "case_narrative": get_case_narrative(text, warnings),
        # criminal-only fields
        "is_penalized": None,
        "punishment_type": None,
        "fine_amount": None,
        "defendant_age": None,
        "defendant_gender": None,
        "education_level": None,
        "has_prior_record": None,
        "is_simplified_procedure": None,
        "primary_article": None,
        "secondary_article": None,
        "extra_articles": None,
        # civil-only fields
        "civil_outcome": None,
        "civil_award_amount": None,
        "parse_warnings": "",
    }

    # Appellate/supervisory rulings review a PRIOR decision rather than
    # deciding fresh - wrong shape for this dataset. Flag, don't force.
    if level != "first_instance":
        warnings.append(f"out_of_scope_{level}")
        row["parse_warnings"] = "|".join(warnings)
        return row

    if category == "criminal":
        age, gender, education = extract_bio(text, warnings)
        prior_record = extract_prior_record(text, warnings)
        simplified = extract_simplified_procedure(text)

        verdict_text = get_verdict_section(text)
        point1_text = get_point1_text(verdict_text)
        is_penalized = classify_outcome(point1_text, warnings)
        punishment_type = classify_punishment_type(verdict_text, is_penalized, warnings)
        primary_art, secondary_art, extra_art = extract_articles(point1_text, warnings)
        fine_amount = extract_fine_amount(verdict_text, punishment_type, warnings)

        row.update({
            "is_penalized": is_penalized,
            "punishment_type": punishment_type,
            "fine_amount": fine_amount,
            "defendant_age": age,
            "defendant_gender": gender,
            "education_level": education,
            "has_prior_record": prior_record,
            "is_simplified_procedure": simplified,
            "primary_article": primary_art,
            "secondary_article": secondary_art,
            "extra_articles": extra_art,
        })

    elif category == "civil":
        verdict_text = get_verdict_section(text)
        point1_text = get_point1_text(verdict_text)
        civil_outcome = classify_civil_outcome(point1_text, warnings)
        civil_award = extract_civil_award(point1_text, warnings)
        row.update({
            "civil_outcome": civil_outcome,
            "civil_award_amount": civil_award,
        })

    elif category == "administrative":
        # Outcome schema not built yet - every real example seen so far
        # has been a procedural dismissal, not a merits ruling, so there's
        # nothing reliable to model outcome from yet. Flagged, not guessed.
        warnings.append("administrative_outcome_not_yet_modeled")

    else:
        warnings.append(f"out_of_scope_{category or 'unknown'}")

    row["parse_warnings"] = "|".join(warnings) if warnings else ""
    return row


def main():
    if len(sys.argv) != 3:
        print("Usage: python3 shuukh_parser.py <input_folder> <output_csv>")
        sys.exit(1)

    input_folder = Path(sys.argv[1])
    output_csv = Path(sys.argv[2])

    files = sorted(p for p in input_folder.iterdir() if p.is_file())
    if not files:
        print(f"No files found in {input_folder}")
        sys.exit(1)

    rows = []
    for f in files:
        try:
            rows.append(parse_file(f))
        except Exception as e:
            print(f"FAILED on {f.name}: {e}", file=sys.stderr)

    if not rows:
        print("Nothing parsed successfully.")
        sys.exit(1)

    fieldnames = list(rows[0].keys())
    with open(output_csv, "w", newline="", encoding="utf-8-sig") as out:
        writer = csv.DictWriter(out, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    flagged = sum(1 for r in rows if r["parse_warnings"])
    print(f"Parsed {len(rows)} file(s) -> {output_csv}")
    print(f"{flagged} row(s) have at least one parse_warning — check those by hand.")


if __name__ == "__main__":
    main()

Usage: python3 shuukh_parser.py <input_folder> <output_csv>


SystemExit: 1

/opt/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
